In [2]:
import urllib.request, json

with urllib.request.urlopen("https://pypi.org/pypi/gptqmodel/json") as r:
    data = json.load(r)

data.keys()

dict_keys(['info', 'last_serial', 'ownership', 'releases', 'urls', 'vulnerabilities'])

In [3]:
data['releases']

{'1.0.1': [{'comment_text': '',
   'digests': {'blake2b_256': '44fab0947cb5302c85a6dde9eb1d78bf3ad9a5261f513a7e48d4ef36cb5df797',
    'md5': 'b4b22b4120f25350d6890e3f1f6a6710',
    'sha256': '5c97e99e949bd9f2c7e787e704c2f1e05a8b5c1fc6cf7eefe4fa0fdcec0e2f8b'},
   'downloads': -1,
   'filename': 'gptqmodel-1.0.1.tar.gz',
   'has_sig': False,
   'md5_digest': 'b4b22b4120f25350d6890e3f1f6a6710',
   'packagetype': 'sdist',
   'python_version': 'source',
   'requires_python': '>=3.8.0',
   'size': 164610,
   'upload_time': '2024-08-15T02:58:05',
   'upload_time_iso_8601': '2024-08-15T02:58:05.248319Z',
   'url': 'https://files.pythonhosted.org/packages/44/fa/b0947cb5302c85a6dde9eb1d78bf3ad9a5261f513a7e48d4ef36cb5df797/gptqmodel-1.0.1.tar.gz',
   'yanked': False,
   'yanked_reason': None}],
 '1.0.2': [{'comment_text': '',
   'digests': {'blake2b_256': '9a3e820cc92613503fe1923245fabcb4dfd1923b9009536e85d4d8a6ed5bcecf',
    'md5': 'd74a9a33b6e02758a5a2c33bca75e959',
    'sha256': '00f102294d2be

In [4]:
import re

def vk(v):
    return tuple(int(x) for x in re.findall(r'\d+', v))

# filter linux x86_64 wheels (manylinux or plain linux), parse out version + py tag + local segment
rows = []
for ver in sorted(data['releases'], key=vk):
    for f in data['releases'][ver]:
        fn = f['filename']
        if not fn.endswith('.whl'):
            continue
        if 'linux_x86_64' not in fn and 'manylinux' not in fn:
            continue
        # wheel filename: pkg-version[+local]-pytag-abitag-platform.whl
        parts = fn.removesuffix('.whl').split('-')
        local = parts[1].split('+', 1)[1] if '+' in parts[1] else ''
        pytag = parts[2] if len(parts) > 2 else ''
        rows.append((ver, pytag, local, fn))

for r in rows:
    print(f'{r[0]:<10} {r[1]:<8} {r[2]:<30} {r[3]}')

In [5]:
# latest-version files only — sometimes wheels live here even when releases[v] is sdist-only
for f in data['urls']:
    print(f['packagetype'], f['filename'])

sdist gptqmodel-7.0.0.tar.gz


In [6]:
# where does gptqmodel point distribution to?
data['info']['project_urls'], data['info'].get('download_url')

({'Homepage': 'https://github.com/ModelCloud/GPTQModel'}, None)

In [7]:
# what fields does info expose, and what does it say the latest version + home_page are?
print('latest version PyPI thinks:', data['info']['version'])
print('home_page:', data['info'].get('home_page'))
print('package_url:', data['info'].get('package_url'))
print()
print('files for latest version:')
for f in data['releases'].get(data['info']['version'], []):
    print(' ', f['packagetype'], f['filename'])

latest version PyPI thinks: 7.0.0
home_page: None
package_url: https://pypi.org/project/GPTQModel/

files for latest version:
  sdist gptqmodel-7.0.0.tar.gz
